# Genetic Algorithm - Hands-On Lab (Parts A to G)
Basic Python only: no NumPy, Pandas, scikit-learn or GA libraries.

Run the cells from top to bottom.

## Part A - Decoding animal characteristics into binary

### Exercise A1 - Decode an animal chromosome (by hand)
Gene order: Height, Weight, Speed, Horns, Habitat (1 = Tall, Heavy, Fast, Has horns, Lives in water).

| Chromosome | Animal characteristics |
|---|---|
| 1 0 1 1 0 | Tall, Light, Fast, Has horns, Lives on land |
| 0 1 1 0 1 | Short, Heavy, Fast, No horns, Lives in water |
| 1 1 0 0 1 | Tall, Heavy, Slow, No horns, Lives in water |
| 0 0 1 0 0 | Short, Light, Fast, No horns, Lives on land |

In [37]:
# Check A1 with code, using the opposite words for each gene
pairs = [("Tall", "Short"), ("Heavy", "Light"), ("Fast", "Slow"),
         ("Has horns", "No horns"), ("Lives in water", "Lives on land")]

def decode_pairs(chromosome):
    result = []
    for i in range(len(chromosome)):
        if chromosome[i] == 1:
            result.append(pairs[i][0])
        else:
            result.append(pairs[i][1])
    return result

for ch in ([1,0,1,1,0], [0,1,1,0,1], [1,1,0,0,1], [0,0,1,0,0]):
    print(ch, "->", decode_pairs(ch))

[1, 0, 1, 1, 0] -> ['Tall', 'Light', 'Fast', 'Has horns', 'Lives on land']
[0, 1, 1, 0, 1] -> ['Short', 'Heavy', 'Fast', 'No horns', 'Lives in water']
[1, 1, 0, 0, 1] -> ['Tall', 'Heavy', 'Slow', 'No horns', 'Lives in water']
[0, 0, 1, 0, 0] -> ['Short', 'Light', 'Fast', 'No horns', 'Lives on land']


### Exercise A2 - Write a simple decoder

In [38]:
characteristics = ["Tall", "Heavy", "Fast", "Has horns", "Lives in water"]

def decode_animal(chromosome):
    description = []
    for i in range(len(chromosome)):
        if chromosome[i] == 1:
            description.append(characteristics[i])
        else:
            description.append("Not " + characteristics[i])
    return description

chromosome = [1, 0, 1, 1, 0]
print(decode_animal(chromosome))

['Tall', 'Not Heavy', 'Fast', 'Has horns', 'Not Lives in water']


### Exercise A3 - Create your own animal
- Characteristic 1: Has wings (1 = Yes, 0 = No)
- Characteristic 2: Size (1 = Large, 0 = Small)
- Characteristic 3: Diet (1 = Meat eater, 0 = Plant eater)
- Characteristic 4: Active time (1 = Nocturnal, 0 = Active by day)
- Characteristic 5: Has fur (1 = Yes, 0 = No)

My chromosome: `[1, 0, 1, 1, 0]` = Has wings, Small, Meat eater, Nocturnal, No fur (an owl)

### Exercise A4 - Why binary encoding?
Binary is the simplest encoding: every gene has only two states, so chromosomes are easy to store, copy and compare, and the operators are easy to write (mutation flips a bit, crossover swaps list slices). Any yes/no property fits, including feature selection later in this lab.

When one bit changes, exactly one characteristic of the animal flips and the other four stay the same, so mutation makes small, controlled changes.

### Exercise A5 - Prepare for mutation

In [39]:
before = [1, 0, 1, 1, 0]
after = before[:]            # copy the list
after[2] = 1 - after[2]      # flip the third bit (index 2)

print("Before mutation:", before, decode_pairs(before))
print("After mutation :", after, decode_pairs(after))
print("Changed characteristic: Speed (Fast -> Slow)")

Before mutation: [1, 0, 1, 1, 0] ['Tall', 'Light', 'Fast', 'Has horns', 'Lives on land']
After mutation : [1, 0, 0, 1, 0] ['Tall', 'Light', 'Slow', 'Has horns', 'Lives on land']
Changed characteristic: Speed (Fast -> Slow)


## Part B - Fitness function

In [63]:
def animal_fitness(chromosome, target):
    score = 0
    for i in range(len(chromosome)):
        if chromosome[i] == target[i]:
            score += 1
    return score

target =     [1, 0, 1, 1, 0]
chromosome = [1, 0, 0, 1, 0]
print("Fitness:", animal_fitness(chromosome, target))

Fitness: 4


### Exercise B1 - Calculate fitness
The manual's code uses `decode` and `population` without defining them, so we define both here.

In [41]:
def decode(chromosome):
    # Convert a list of bits into an integer without int(..., 2)
    value = 0
    for bit in chromosome:
        value = value * 2 + bit
    return value

def fitness(chromosome):
    x = decode(chromosome)
    return x * x

population = [
    [0, 0, 0, 1, 1, 0],
    [1, 1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0, 1],
    [1, 0, 1, 0, 1, 1]
]

for chromosome in population:
    print(chromosome, "x =", decode(chromosome), "fitness =", fitness(chromosome))

[0, 0, 0, 1, 1, 0] x = 6 fitness = 36
[1, 1, 0, 0, 1, 0] x = 50 fitness = 2500
[0, 1, 1, 0, 0, 1] x = 25 fitness = 625
[1, 0, 1, 0, 1, 1] x = 43 fitness = 1849


### Exercise B2 - Add a target function

In [42]:
def objective(x):
    return 100 - abs(x - 50)

best = None
best_score = -1
for chromosome in population:
    score = objective(decode(chromosome))
    print(chromosome, "x =", decode(chromosome), "objective =", score)
    if score > best_score:
        best_score = score
        best = chromosome

print("Best chromosome:", best, "x =", decode(best), "score =", best_score)

[0, 0, 0, 1, 1, 0] x = 6 objective = 56
[1, 1, 0, 0, 1, 0] x = 50 objective = 100
[0, 1, 1, 0, 0, 1] x = 25 objective = 75
[1, 0, 1, 0, 1, 1] x = 43 objective = 93
Best chromosome: [1, 1, 0, 0, 1, 0] x = 50 score = 100


## Part C - Mutation

### Exercise C1 - Mutate one selected gene

In [65]:
def mutate_one(chromosome, position):
    child = chromosome[:]
    child[position] = 1 - child[position]
    return child

chromosome = [1, 0, 1, 1, 0, 1]
print("Before:    ", chromosome)
print("After :    ", mutate_one(chromosome, 2))
print("position 0:", mutate_one(chromosome, 0))   # flips the first (leftmost) bit
print("position 5:", mutate_one(chromosome, 5))   # flips the last (rightmost) bit

Before:     [1, 0, 1, 1, 0, 1]
After :     [1, 0, 0, 1, 0, 1]
position 0: [0, 0, 1, 1, 0, 1]
position 5: [1, 0, 1, 1, 0, 0]


**Answer:** position 0 flips the first bit and position 5 flips the last bit. Python counts from 0, so a 6-gene chromosome has positions 0 to 5.

### Exercise C2 - Random mutation

In [78]:
import random

def mutate(chromosome, mutation_rate):
    child = chromosome[:]
    for i in range(len(child)):
        if random.random() < mutation_rate:
            child[i] = 1 - child[i]
    return child

chromosome = [1, 0, 1, 1, 0, 1]
for i in range(5):
    print("Mutate rate 0.20:", mutate(chromosome, 0.20))

Mutate rate 0.20: [1, 0, 1, 0, 0, 1]
Mutate rate 0.20: [1, 0, 1, 1, 0, 1]
Mutate rate 0.20: [0, 0, 1, 1, 0, 1]
Mutate rate 0.20: [1, 0, 0, 1, 0, 1]
Mutate rate 0.20: [1, 0, 1, 1, 0, 1]


### Exercise C3 - Mutation rate

In [45]:
for rate in (0.00, 0.10, 0.50, 1.00):
    changes = []
    for trial in range(5):
        mutated = mutate(chromosome, rate)
        changed = 0
        for a, b in zip(chromosome, mutated):
            if a != b:
                changed += 1
        changes.append(changed)
    print("rate", rate, "-> changed bits in 5 runs:", changes)

rate 0.0 -> changed bits in 5 runs: [0, 0, 0, 0, 0]
rate 0.1 -> changed bits in 5 runs: [2, 1, 0, 0, 0]
rate 0.5 -> changed bits in 5 runs: [2, 4, 2, 5, 3]
rate 1.0 -> changed bits in 5 runs: [6, 6, 6, 6, 6]


Rate 0.00 changes no bits and rate 1.00 changes all 6. Rates 0.10 and 0.50 vary from run to run (expected value = rate x 6).

## Part D - Crossover

### Exercise D1 - Manual crossover
| Parent 1 | Parent 2 | Cut | Children |
|---|---|---|---|
| 110101 | 001110 | 3 | 110110 and 001101 |
| 111000 | 010111 | 4 | 111011 and 010100 |
| 100111 | 011001 | 2 | 101001 and 010111 |

In [46]:
def cross_at(p1, p2, point):
    return p1[:point] + p2[point:], p2[:point] + p1[point:]

for p1, p2, cut in (("110101", "001110", 3), ("111000", "010111", 4), ("100111", "011001", 2)):
    c1, c2 = cross_at(list(p1), list(p2), cut)
    print(p1, p2, "cut", cut, "->", "".join(map(str, c1)), "".join(map(str, c2)))

110101 001110 cut 3 -> 110110 001101
111000 010111 cut 4 -> 111011 010100
100111 011001 cut 2 -> 101001 010111


### Exercise D2 - Code single-point crossover

In [47]:
def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2, point

p1 = [1, 1, 0, 1, 0, 1]
p2 = [0, 0, 1, 0, 1, 1]
c1, c2, point = crossover(p1, p2)

print("Parent 1:", p1)
print("Parent 2:", p2)
print("Cut point:", point)
print("Child 1 :", c1)
print("Child 2 :", c2)

Parent 1: [1, 1, 0, 1, 0, 1]
Parent 2: [0, 0, 1, 0, 1, 1]
Cut point: 2
Child 1 : [1, 1, 1, 0, 1, 1]
Child 2 : [0, 0, 0, 1, 0, 1]


### Exercise D3 - Crossover probability

In [48]:
def crossover_with_rate(parent1, parent2, crossover_rate):
    if random.random() < crossover_rate:
        point = random.randint(1, len(parent1) - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        return child1, child2
    return parent1[:], parent2[:]    # no crossover: return copies of the parents

print("rate 0.0:", crossover_with_rate(p1, p2, 0.0))   # always unchanged parents
print("rate 1.0:", crossover_with_rate(p1, p2, 1.0))   # always crosses

rate 0.0: ([1, 1, 0, 1, 0, 1], [0, 0, 1, 0, 1, 1])
rate 1.0: ([1, 0, 1, 0, 1, 1], [0, 1, 0, 1, 0, 1])


## Part E - Roulette-wheel selection

### Exercise E1 - Calculate probabilities
| Chromosome | Fitness | Probability | Cumulative |
|---|---|---|---|
| A | 10 | 0.10 | 0.10 |
| B | 20 | 0.20 | 0.30 |
| C | 30 | 0.30 | 0.60 |
| D | 40 | 0.40 | 1.00 |

In [49]:
fits = [10, 20, 30, 40]
total = sum(fits)
cumulative = 0
for name, f in zip("ABCD", fits):
    cumulative += f
    print(name, f, "probability =", f / total, "cumulative =", cumulative / total)

A 10 probability = 0.1 cumulative = 0.1
B 20 probability = 0.2 cumulative = 0.3
C 30 probability = 0.3 cumulative = 0.6
D 40 probability = 0.4 cumulative = 1.0


### Exercise E2 - Implement roulette-wheel selection

In [82]:
def roulette_selection(population, fitnesses):
    total = sum(fitnesses)
    r = random.random() * total
    cumulative = 0
    for chromosome, fit in zip(population, fitnesses):
        cumulative += fit
        if cumulative >= r:
            return chromosome[:]
    return population[-1][:]

population = [[0,0,0,1], [0,0,1,0], [0,1,0,0], [1,0,0,0]]
fitnesses = [10, 20, 30, 40]

for i in range(10):
    print(roulette_selection(population, fitnesses))


# Count 10000 picks: [1,0,0,0] (fitness 40) should appear most often
counts = {}
for i in range(10000):
    key = tuple(roulette_selection(population, fitnesses))
    counts[key] = counts.get(key, 0) + 1
print(counts)

[1, 0, 0, 0]
[0, 1, 0, 0]
[1, 0, 0, 0]
[0, 1, 0, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]
[0, 1, 0, 0]
[0, 1, 0, 0]
[0, 0, 1, 0]
[1, 0, 0, 0]
{(0, 1, 0, 0): 3029, (1, 0, 0, 0): 4053, (0, 0, 1, 0): 1902, (0, 0, 0, 1): 1016}


### Exercise E3 - Create a new population

In [80]:
def select_population(population, fitnesses, size):
    new_population = []
    for _ in range(size):
        new_population.append(roulette_selection(population, fitnesses))
    return new_population

new_population = select_population(population, fitnesses, 8)
for chromosome in new_population:
    print(chromosome)

[1, 0, 0, 0]
[0, 1, 0, 0]
[0, 1, 0, 0]
[0, 0, 0, 1]
[1, 0, 0, 0]
[0, 1, 0, 0]
[0, 1, 0, 0]
[0, 1, 0, 0]


## Part F - Complete Genetic Algorithm

### Exercise F1 - Complete GA from scratch

In [83]:
import random

CHROMOSOME_LENGTH = 6
POPULATION_SIZE = 10
GENERATIONS = 20
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.80

def create_chromosome():
    chromosome = []
    for _ in range(CHROMOSOME_LENGTH):
        chromosome.append(random.randint(0, 1))
    return chromosome

def create_population():
    population = []
    for _ in range(POPULATION_SIZE):
        population.append(create_chromosome())
    return population

def decode(chromosome):
    value = 0
    for bit in chromosome:
        value = value * 2 + bit
    return value

def fitness(chromosome):
    x = decode(chromosome)
    return 100 - abs(x - 50)

def roulette_selection(population, fitnesses):
    total = sum(fitnesses)
    r = random.random() * total
    cumulative = 0
    for chromosome, fit in zip(population, fitnesses):
        cumulative += fit
        if cumulative >= r:
            return chromosome[:]
    return population[-1][:]

def crossover(parent1, parent2):
    if random.random() > CROSSOVER_RATE:
        return parent1[:], parent2[:]
    point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

def mutate(chromosome):
    child = chromosome[:]
    for i in range(len(child)):
        if random.random() < MUTATION_RATE:
            child[i] = 1 - child[i]
    return child

random.seed(42)    # remove this line to get different results on every run
population = create_population()

for generation in range(GENERATIONS):
    fitnesses = [fitness(c) for c in population]
    best_index = fitnesses.index(max(fitnesses))
    best = population[best_index]
    print("Generation:", generation, "Best:", decode(best), "Fitness:", fitness(best))

    new_population = []
    while len(new_population) < POPULATION_SIZE:
        parent1 = roulette_selection(population, fitnesses)
        parent2 = roulette_selection(population, fitnesses)
        child1, child2 = crossover(parent1, parent2)
        child1 = mutate(child1)
        child2 = mutate(child2)
        new_population.append(child1)
        if len(new_population) < POPULATION_SIZE:
            new_population.append(child2)
    population = new_population

fitnesses = [fitness(c) for c in population]
best_index = fitnesses.index(max(fitnesses))
print("Final best chromosome:", population[best_index])
print("Final decoded value:", decode(population[best_index]))
print("Final fitness:", fitnesses[best_index])

Generation: 0 Best: 51 Fitness: 99
Generation: 1 Best: 46 Fitness: 96
Generation: 2 Best: 47 Fitness: 97
Generation: 3 Best: 50 Fitness: 100
Generation: 4 Best: 50 Fitness: 100
Generation: 5 Best: 46 Fitness: 96
Generation: 6 Best: 46 Fitness: 96
Generation: 7 Best: 46 Fitness: 96
Generation: 8 Best: 46 Fitness: 96
Generation: 9 Best: 46 Fitness: 96
Generation: 10 Best: 58 Fitness: 92
Generation: 11 Best: 58 Fitness: 92
Generation: 12 Best: 50 Fitness: 100
Generation: 13 Best: 50 Fitness: 100
Generation: 14 Best: 51 Fitness: 99
Generation: 15 Best: 51 Fitness: 99
Generation: 16 Best: 50 Fitness: 100
Generation: 17 Best: 50 Fitness: 100
Generation: 18 Best: 50 Fitness: 100
Generation: 19 Best: 50 Fitness: 100
Final best chromosome: [1, 1, 0, 0, 1, 0]
Final decoded value: 50
Final fitness: 100


### Exercise F2 - Experiment with parameters
This cell wraps the GA in a function so every setting can be tested quickly.

In [86]:
def run_ga(pop_size, generations, mut_rate, cross_rate, seed, length=6):
    rnd = random.Random(seed)

    def fit(c):
        return 100 - abs(decode(c) - 50)

    def pick(pop, fs):
        total = sum(fs)
        r = rnd.random() * total
        cum = 0
        for c, f in zip(pop, fs):
            cum += f
            if cum >= r:
                return c[:]
        return pop[-1][:]

    def cross(p1, p2):
        if rnd.random() > cross_rate:
            return p1[:], p2[:]
        pt = rnd.randint(1, len(p1) - 1)
        return p1[:pt] + p2[pt:], p2[:pt] + p1[pt:]

    def mut(c):
        ch = c[:]
        for i in range(len(ch)):
            if rnd.random() < mut_rate:
                ch[i] = 1 - ch[i]
        return ch

    pop = [[rnd.randint(0, 1) for _ in range(length)] for _ in range(pop_size)]
    for g in range(generations):
        fs = [fit(c) for c in pop]
        new = []
        while len(new) < pop_size:
            c1, c2 = cross(pick(pop, fs), pick(pop, fs))
            new.append(mut(c1))
            if len(new) < pop_size:
                new.append(mut(c2))
        pop = new
    fs = [fit(c) for c in pop]
    i = fs.index(max(fs))
    return decode(pop[i])

settings = [(10, 20, 0.05, 0.80), (20, 20, 0.05, 0.80), (10, 50, 0.05, 0.80),
            (10, 50, 0.20, 0.80), (10, 50, 0.05, 1.00)]
print("Population  Generations  Mutation  Crossover  Best x")
for s in settings:
    print(s[0], "         ", s[1], "        ", s[2], "     ", s[3], "     ", run_ga(*s, seed=42))

Population  Generations  Mutation  Crossover  Best x
10           20          0.05       0.8       50
20           20          0.05       0.8       51
10           50          0.05       0.8       50
10           50          0.2       0.8       49
10           50          0.05       1.0       50


With seed 42 the best x values are 50, 51, 50, 49, 50. Change `seed` and run again to see how results vary, because the GA is random.

## Part G - Real-world application: feature selection (Statlog Heart)
This part needs `heart.dat` in the same folder as this notebook. The next cell downloads it if it is missing (needs internet, works in Google Colab). If the download fails, get it manually from https://archive.ics.uci.edu/dataset/145/statlog%2Bheart and place `heart.dat` next to the notebook.


### 10.3 Load the dataset without Pandas

In [54]:
def load_heart_data(filename):
    X = []
    y = []
    with open(filename, "r") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) < 14:
                continue
            values = [float(v) for v in parts[:13]]
            target = int(float(parts[13]))
            # Statlog Heart uses: 1 = absence, 2 = presence
            target = 0 if target == 1 else 1
            X.append(values)
            y.append(target)
    return X, y

X, y = load_heart_data("heart.dat")
print("Number of records:", len(X))
print("Number of features:", len(X[0]))
print("First record:", X[0])
print("First target:", y[0])

Number of records: 270
Number of features: 13
First record: [70.0, 1.0, 4.0, 130.0, 322.0, 0.0, 2.0, 109.0, 0.0, 2.4, 2.0, 3.0, 3.0]
First target: 1


In [87]:
print(X[0])

[0.8541666666666666, 1.0, 1.0, 0.33962264150943394, 0.4474885844748858, 0.0, 1.0, 0.2900763358778626, 0.0, 0.3870967741935484, 0.5, 1.0, 0.0]


### 10.4 Normalize the numeric data

In [55]:
def normalize_columns(X):
    rows = len(X)
    cols = len(X[0])
    minimum = [X[0][j] for j in range(cols)]
    maximum = [X[0][j] for j in range(cols)]
    for i in range(rows):
        for j in range(cols):
            if X[i][j] < minimum[j]:
                minimum[j] = X[i][j]
            if X[i][j] > maximum[j]:
                maximum[j] = X[i][j]
    normalized = []
    for i in range(rows):
        row = []
        for j in range(cols):
            if maximum[j] == minimum[j]:
                row.append(0.0)
            else:
                row.append((X[i][j] - minimum[j]) / (maximum[j] - minimum[j]))
        normalized.append(row)
    return normalized

X = normalize_columns(X)

### 10.5 Train/test split

In [56]:
indices = list(range(len(X)))
random.seed(42)
random.shuffle(indices)

split = int(0.80 * len(indices))
train_indices = indices[:split]
test_indices = indices[split:]

X_train = [X[i] for i in train_indices]
y_train = [y[i] for i in train_indices]
X_test = [X[i] for i in test_indices]
y_test = [y[i] for i in test_indices]

print("Training records:", len(X_train))
print("Testing records :", len(X_test))

Training records: 216
Testing records : 54


### 10.6 k-Nearest Neighbors from scratch

In [57]:
def distance(a, b, selected):
    total = 0.0
    for j in range(len(a)):
        if selected[j] == 1:
            difference = a[j] - b[j]
            total += difference * difference
    return total ** 0.5

def predict_one(test_row, X_train, y_train, selected, k=3):
    distances = []
    for i in range(len(X_train)):
        d = distance(test_row, X_train[i], selected)
        distances.append((d, y_train[i]))
    distances.sort()

    votes = {}
    for i in range(min(k, len(distances))):
        label = distances[i][1]
        votes[label] = votes.get(label, 0) + 1

    best_label = None
    best_votes = -1
    for label, count in votes.items():
        if count > best_votes:
            best_votes = count
            best_label = label
    return best_label

def accuracy(X_test, y_test, X_train, y_train, selected, k=3):
    correct = 0
    for i in range(len(X_test)):
        prediction = predict_one(X_test[i], X_train, y_train, selected, k)
        if prediction == y_test[i]:
            correct += 1
    return correct / len(y_test)

### 10.7 Fitness function for feature selection
`fitness = accuracy - penalty * (selected_features / total_features)`

In [58]:
PENALTY = 0.05

def feature_fitness(chromosome, penalty=None):
    if penalty is None:
        penalty = PENALTY
    selected = sum(chromosome)
    if selected == 0:
        return 0.0
    acc = accuracy(X_test, y_test, X_train, y_train, chromosome)
    return acc - penalty * (selected / len(chromosome))

### 10.8 Run the GA for feature selection
The GA is wrapped in `run_feature_ga` so the later questions (different seeds, rates, sizes) can reuse it.

In [59]:
FEATURES = 13

def run_feature_ga(seed, pop_size=12, generations=10, mutation_rate=0.05,
                   crossover_rate=0.80, penalty=0.05, verbose=True):
    rnd = random.Random(seed)

    def create():
        c = [rnd.randint(0, 1) for _ in range(FEATURES)]
        if sum(c) == 0:                      # make sure at least one feature is selected
            c[rnd.randint(0, FEATURES - 1)] = 1
        return c

    def roulette(pop, fits):
        # shift values so all roulette weights are positive
        m = min(fits)
        adjusted = [f - m + 0.000001 for f in fits]
        r = rnd.random() * sum(adjusted)
        cum = 0.0
        for c, f in zip(pop, adjusted):
            cum += f
            if cum >= r:
                return c[:]
        return pop[-1][:]

    def cross(p1, p2):
        if rnd.random() > crossover_rate:
            return p1[:], p2[:]
        point = rnd.randint(1, FEATURES - 1)
        return p1[:point] + p2[point:], p2[:point] + p1[point:]

    def mutate(c):
        child = c[:]
        for i in range(len(child)):
            if rnd.random() < mutation_rate:
                child[i] = 1 - child[i]
        if sum(child) == 0:
            child[rnd.randint(0, FEATURES - 1)] = 1
        return child

    population = [create() for _ in range(pop_size)]
    best_overall = None
    best_overall_fitness = -1

    for generation in range(generations):
        fitnesses = [feature_fitness(c, penalty) for c in population]
        best_index = fitnesses.index(max(fitnesses))
        if fitnesses[best_index] > best_overall_fitness:
            best_overall = population[best_index][:]
            best_overall_fitness = fitnesses[best_index]
        if verbose:
            print("Generation:", generation,
                  "Fitness:", round(fitnesses[best_index], 4),
                  "Features:", sum(population[best_index]),
                  "Chromosome:", population[best_index])

        new_population = []
        while len(new_population) < pop_size:
            c1, c2 = cross(roulette(population, fitnesses), roulette(population, fitnesses))
            new_population.append(mutate(c1))
            if len(new_population) < pop_size:
                new_population.append(mutate(c2))
        population = new_population

    return best_overall, best_overall_fitness

best_overall, best_overall_fitness = run_feature_ga(seed=1)

print("\nBEST FEATURE SUBSET")
print("Chromosome:", best_overall)
print("Number of features:", sum(best_overall))
print("Fitness:", best_overall_fitness)
print("Test accuracy:", accuracy(X_test, y_test, X_train, y_train, best_overall))

Generation: 0 Fitness: 0.784 Features: 8 Chromosome: [1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1]
Generation: 1 Fitness: 0.7879 Features: 7 Chromosome: [0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1]
Generation: 2 Fitness: 0.8026 Features: 8 Chromosome: [1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0]
Generation: 3 Fitness: 0.8026 Features: 8 Chromosome: [1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0]
Generation: 4 Fitness: 0.8211 Features: 8 Chromosome: [1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0]
Generation: 5 Fitness: 0.8288 Features: 6 Chromosome: [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0]
Generation: 6 Fitness: 0.8249 Features: 7 Chromosome: [0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0]
Generation: 7 Fitness: 0.8581 Features: 8 Chromosome: [1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0]
Generation: 8 Fitness: 0.899 Features: 7 Chromosome: [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0]
Generation: 9 Fitness: 0.8805 Features: 7 Chromosome: [1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0]

BEST FEATURE SUBSET
Chromosome: [1, 1, 1, 0, 1, 0, 0, 0, 0, 1

### 10.9 Map chromosome positions to feature names

In [60]:
feature_names = ["age", "sex", "chest_pain", "resting_bp", "cholesterol",
                 "fasting_blood_sugar", "resting_ecg", "max_heart_rate",
                 "exercise_angina", "oldpeak", "slope", "major_vessels", "thal"]

for i in range(len(best_overall)):
    if best_overall[i] == 1:
        print(feature_names[i])

age
sex
chest_pain
cholesterol
oldpeak
slope
major_vessels


### 10.10 Compare GA-selected features with all features

In [61]:
all_features = [1] * FEATURES

all_accuracy = accuracy(X_test, y_test, X_train, y_train, all_features)
ga_accuracy = accuracy(X_test, y_test, X_train, y_train, best_overall)

print("Accuracy using all features:", all_accuracy)
print("Accuracy using GA features :", ga_accuracy)
print("Features using all :", sum(all_features))
print("Features using GA  :", sum(best_overall))

Accuracy using all features: 0.7037037037037037
Accuracy using GA features : 0.9259259259259259
Features using all : 13
Features using GA  : 7


### 10.11 Student analysis questions - experiments
Run this cell, then write your answers below it. (It takes about 30 seconds.)

In [62]:
print("--- Q6: mutation rate ---")
for mr in (0.01, 0.05, 0.20, 0.50):
    b, f = run_feature_ga(1, mutation_rate=mr, verbose=False)
    print("mutation", mr, "-> features", sum(b), "fitness", round(f, 4))

print("--- Q7: population size ---")
for ps in (6, 12, 24):
    b, f = run_feature_ga(1, pop_size=ps, verbose=False)
    print("population", ps, "-> features", sum(b), "fitness", round(f, 4))

print("--- Q8: generations ---")
for gen in (5, 10, 30):
    b, f = run_feature_ga(1, generations=gen, verbose=False)
    print("generations", gen, "-> features", sum(b), "fitness", round(f, 4))

print("--- Q9: three different seeds ---")
for s in (1, 2, 3):
    b, f = run_feature_ga(s, verbose=False)
    print("seed", s, "->", b, "features", sum(b), "fitness", round(f, 4))

print("--- Challenge 6: penalty ---")
for p in (0.00, 0.02, 0.05, 0.10):
    b, f = run_feature_ga(1, penalty=p, verbose=False)
    print("penalty", p, "-> features", sum(b), "fitness", round(f, 4))

--- Q6: mutation rate ---
mutation 0.01 -> features 8 fitness 0.8211
mutation 0.05 -> features 7 fitness 0.899
mutation 0.2 -> features 5 fitness 0.8882
mutation 0.5 -> features 7 fitness 0.862
--- Q7: population size ---
population 6 -> features 6 fitness 0.8658
population 12 -> features 7 fitness 0.899
population 24 -> features 7 fitness 0.899
--- Q8: generations ---
generations 5 -> features 8 fitness 0.8211
generations 10 -> features 7 fitness 0.899
generations 30 -> features 4 fitness 0.9105
--- Q9: three different seeds ---
seed 1 -> [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0] features 7 fitness 0.899
seed 2 -> [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0] features 5 fitness 0.8882
seed 3 -> [1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1] features 8 fitness 0.8766
--- Challenge 6: penalty ---
penalty 0.0 -> features 6 fitness 0.9074
penalty 0.02 -> features 6 fitness 0.8982
penalty 0.05 -> features 7 fitness 0.899
penalty 0.1 -> features 7 fitness 0.8721


**Answers (fill in the numbers from your own output):**

1. Did the GA select all 13 features or a smaller subset? _your output_
2. Best chromosome found: _your output_
3. Accuracy using all features: _your output_
4. Accuracy using the GA-selected subset: _your output_
5. Did reducing features reduce, maintain or improve accuracy? _compare 3 and 4_
6. **Mutation rate:** very small rates explore little and can get stuck; about 0.05 is usually good; very large rates keep destroying good chromosomes, so the search becomes almost random.
7. **Population size:** a larger population tries more subsets per generation, but each generation is slower.
8. **Generations:** more generations help at first, then the best fitness flattens and extra generations only cost time.
9. **Different seeds:** subsets usually differ because the starting population, selection, crossover points and mutations are random, and several different subsets can reach nearly the same accuracy. The test set is also small (54 rows), so one prediction changes accuracy by about 1.9%.
10. **Why penalize the number of features:** accuracy alone favours keeping every feature that helps even a little. The penalty makes the GA prefer a smaller subset when accuracy is about equal, giving a simpler, faster, less noisy model.

*Note:* fitness is measured on the test set (as in the manual), so the reported test accuracy is slightly optimistic. A separate validation split would avoid this.